# NB_PERF_02 · Medir las consultas DAX con traza

**Paso 2 del framework de performance. Solo lectura** sobre los datos: lo único que hace en el modelo es **vaciar su caché** antes de cada repetición en frío.

Corre cada consulta de `consultas.csv` varias veces, en **frío** (caché vaciada) y en **caliente**, con una traza del motor como la de DAX Studio, y guarda por ejecución:

| Métrica | Qué es | Qué sugiere si es alta |
|---|---|---|
| `total_ms` | Duración de `QueryEnd` | |
| `se_ms` | Suma de `VertiPaqSEQueryEnd` (sin las internas) | Modelo: columnas pesadas, relaciones, cardinalidad, falta de agregaciones |
| `fe_ms` | `total_ms - se_ms` | DAX: iteradores, callbacks, cálculos que no se empujan al motor de almacenamiento |
| `se_consultas` | Cantidad de consultas al motor de almacenamiento | DAX que fragmenta el plan (muchas consultas chicas) |
| `dq_consultas`, `dq_ms` | Consultas DirectQuery | Fallback de Direct Lake o tablas DirectQuery |
| `huella` | Hash del resultado | Si cambia entre `antes` y `despues`, la optimización cambió la respuesta |

`consultas.csv` vive en `Files/doc_modelos/<modelo>/perf/consultas.csv` y es **el mismo para todas las corridas**: así `antes` y `despues` se comparan con la misma vara. Plantilla en `skill/optimizar-modelo-semantico/plantillas/consultas.csv`.

> Las trazas de semantic-link son, según su documentación, de uso exploratorio y con tiempos que dependen de la carga del servidor. Por eso se repite cada consulta y se compara la **mediana**, en la misma capacidad y en horarios parecidos.

In [ ]:
# Parámetros
MODELO = "<nombre-del-modelo-semantico>"
WORKSPACE = "<workspace-de-prueba>"
CORRIDA = "antes"                        # la misma que usaste en NB_PERF_01
COMMIT = ""
CARPETA_BASE = "/lakehouse/default/Files/doc_modelos"
REPETICIONES = 5                         # por consulta y por modo
MODOS = ["frio", "caliente"]
PAUSA_SEG = 1                            # pausa entre ejecuciones
SEG_ESPERA_TRAZA = 10                    # espera final para que lleguen los eventos
DEPENDENCIAS_TOP = 5                     # columnas de las N consultas más lentas (0 = no calcular)
SOBREESCRIBIR = False
ESQUEMA_DELTA = "perf"                  # esquema del lakehouse para las tablas Delta
GUARDAR_DELTA = True

In [ ]:
%pip install -q semantic-link-labs==0.17.1

In [ ]:
# --- funciones (sin dependencias de Fabric: las prueba pruebas/prueba_local.py)
import hashlib
import re

import pandas as pd

COLUMNAS_CONSULTAS = ["id", "nombre", "origen", "dax", "activa"]

# Eventos y columnas de la traza. Mismo esquema que usa semantic-link-labs para su
# medidor de DAX (dax_perf_optimizer), con RequestID para agrupar los eventos de cada consulta.
EVENTOS_TRAZA = {
    "QueryBegin": ["EventClass", "EventSubclass", "TextData", "StartTime", "RequestID"],
    "QueryEnd": ["EventClass", "EventSubclass", "TextData", "StartTime", "EndTime",
                 "Duration", "CpuTime", "Success", "RequestID"],
    "VertiPaqSEQueryEnd": ["EventClass", "EventSubclass", "TextData", "StartTime", "EndTime",
                           "Duration", "CpuTime", "RequestID"],
    "VertiPaqSEQueryCacheMatch": ["EventClass", "EventSubclass", "TextData", "RequestID"],
    "DirectQueryEnd": ["EventClass", "TextData", "StartTime", "EndTime", "Duration", "CpuTime", "RequestID"],
}


def validar_consultas(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Devuelve las consultas activas y la lista de problemas (vacía si está todo bien)."""
    problemas = []
    faltan = [c for c in COLUMNAS_CONSULTAS if c not in df.columns]
    if faltan:
        return df.iloc[0:0], [f"Faltan columnas: {faltan}"]
    df = df.fillna("")
    activas = df[df["activa"].str.strip().str.lower().isin(["si", "sí", "true", "1"])].copy()
    if activas.empty:
        problemas.append("No hay consultas con activa = si")
    dup = activas["id"][activas["id"].duplicated()].tolist()
    if dup:
        problemas.append(f"IDs repetidos: {dup}")
    for _, r in activas.iterrows():
        if not re.match(r"^\s*(DEFINE|EVALUATE)\b", r["dax"], re.IGNORECASE):
            problemas.append(f"{r['id']}: la consulta tiene que empezar con DEFINE o EVALUATE")
        if not re.match(r"^[A-Za-z0-9_\-]+$", r["id"]):
            problemas.append(f"{r['id']}: el id solo puede tener letras, números, - y _")
    return activas, problemas


def marca(id_consulta: str, rep: int, modo: str) -> str:
    return f"perf:{id_consulta}:{modo}:{rep}"


def marcar(dax: str, m: str) -> str:
    """Comentario al principio de la consulta: aparece en TextData y permite encontrarla en la traza."""
    return f"/* {m} */\n{dax}"


def _col(df: pd.DataFrame, *nombres: str):
    for n in nombres:
        if n in df.columns:
            return n
    return None


def _int(v) -> int:
    return 0 if v is None or pd.isna(v) else int(v)


def metricas_de_traza(logs: pd.DataFrame, marcas: list[str]) -> pd.DataFrame:
    """Una fila por marca con las métricas al estilo DAX Studio.

    Agrupa por RequestID. Si la traza no trae RequestID, usa la ventana de tiempo de la consulta.
    """
    ec = _col(logs, "Event Class", "EventClass")
    es = _col(logs, "Event Subclass", "EventSubclass")
    td = _col(logs, "Text Data", "TextData")
    rq = _col(logs, "Request ID", "RequestID")
    st = _col(logs, "Start Time", "StartTime")
    et = _col(logs, "End Time", "EndTime")
    du = _col(logs, "Duration")
    cpu = _col(logs, "Cpu Time", "CPU Time", "CpuTime")
    filas = []
    for m in marcas:
        fin = logs[(logs[ec] == "QueryEnd") & logs[td].astype(str).str.contains(m, regex=False)]
        if fin.empty:
            filas.append({"marca": m, "encontrada": False})
            continue
        q = fin.iloc[-1]
        if rq and pd.notna(q[rq]) and str(q[rq]):
            ev = logs[logs[rq] == q[rq]]
        else:
            ev = logs[(logs[st] >= q[st]) & (logs[st] <= q[et])]
        se = ev[ev[ec] == "VertiPaqSEQueryEnd"]
        se_ext = se[~se[es].astype(str).str.contains("Internal", case=False, na=False)] if es else se
        dq = ev[ev[ec] == "DirectQueryEnd"]
        total = _int(q[du])
        se_ms = int(se_ext[du].fillna(0).sum())
        filas.append({
            "marca": m, "encontrada": True,
            "total_ms": total, "se_ms": se_ms, "fe_ms": max(total - se_ms, 0),
            "cpu_ms": _int(q[cpu]) if cpu else None,
            "se_consultas": len(se_ext),
            "se_cache_hits": int((ev[ec] == "VertiPaqSEQueryCacheMatch").sum()),
            "dq_consultas": len(dq), "dq_ms": int(dq[du].fillna(0).sum()) if len(dq) else 0,
        })
    return pd.DataFrame(filas)


def huella_resultado(df: pd.DataFrame, decimales: int = 6) -> str:
    """Hash del resultado, independiente del orden de filas y columnas. Redondea los decimales
    para no marcar como distinto un cambio de 1e-12 por el orden de suma."""
    if df is None or df.empty:
        return "vacio"
    d = df.copy()
    d.columns = [re.sub(r"^.*\[(.*)\]$", r"\1", str(c)) for c in d.columns]
    d = d[sorted(d.columns)]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(decimales)
    texto = d.astype(str).sort_values(list(d.columns)).to_csv(index=False)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()[:16]


def resumir(mediciones: pd.DataFrame) -> pd.DataFrame:
    """Mediana, mínimo y p90 por consulta y modo. La huella tiene que ser única por consulta."""
    ok = mediciones[mediciones["encontrada"] == True]
    g = ok.groupby(["id", "modo"])
    res = g.agg(
        n=("total_ms", "size"),
        total_ms_mediana=("total_ms", "median"), total_ms_min=("total_ms", "min"),
        total_ms_p90=("total_ms", lambda s: s.quantile(0.9)),
        se_ms_mediana=("se_ms", "median"), fe_ms_mediana=("fe_ms", "median"),
        se_consultas_mediana=("se_consultas", "median"), dq_consultas_max=("dq_consultas", "max"),
    ).reset_index()
    huellas = mediciones.groupby("id")["huella"].agg(lambda s: "|".join(sorted(set(s.dropna()))))
    res["huella"] = res["id"].map(huellas)
    res["huella_estable"] = ~res["huella"].str.contains("|", regex=False)
    return res

## 1. Leer y validar `consultas.csv`

Cómo conseguir consultas representativas, de mejor a peor:

1. **Performance Analyzer** de Power BI Desktop (o del servicio) sobre las páginas que la gente usa: *Copy query* de cada visual lento. Poné `origen = performance_analyzer:<página>/<visual>`.
2. **Workspace monitoring** de Fabric, si está habilitado: las consultas reales más lentas o más frecuentes.
3. Escritas a mano para las medidas críticas.

No pongas filtros por personas ni datos sensibles en el texto de la consulta: el CSV y la traza quedan en el lakehouse.

In [ ]:
import json
import os
import time
from datetime import datetime, timezone

import sempy.fabric as fabric
import sempy_labs as labs

base_modelo = CARPETA_BASE.rstrip("/") + "/" + re.sub(r"[^\w\-]+", "_", MODELO)
salida = f"{base_modelo}/perf/{CORRIDA}"
if os.path.exists(f"{salida}/mediciones.csv") and not SOBREESCRIBIR:
    raise FileExistsError(f"La corrida '{CORRIDA}' ya tiene mediciones. Usá otro nombre o SOBREESCRIBIR = True.")
os.makedirs(salida, exist_ok=True)

consultas, problemas = validar_consultas(
    pd.read_csv(f"{base_modelo}/perf/consultas.csv", dtype=str, keep_default_na=False))
if problemas:
    raise ValueError("Revisá consultas.csv:\n- " + "\n- ".join(problemas))
print(f"{len(consultas)} consultas × {len(MODOS)} modos × {REPETICIONES} repeticiones")

## 2. Ejecutar con la traza

Una sola traza para toda la corrida. Cada ejecución lleva un comentario `/* perf:<id>:<modo>:<rep> */` para encontrarla en los eventos. En modo frío se vacía la caché del modelo antes de cada repetición con `labs.clear_cache` (comando XMLA `ClearCache`).

En **Direct Lake**, `ClearCache` no saca las columnas de memoria: la primera consulta después de un reframe o de un período sin uso puede ser más lenta. Compará siempre corridas en el mismo estado (por ejemplo, las dos después de refrescar el modelo).

In [ ]:
ejecuciones = []      # (id, modo, rep, marca, filas, huella, error)

with fabric.create_trace_connection(MODELO, WORKSPACE) as conexion:
    with conexion.create_trace(EVENTOS_TRAZA, f"perf_{CORRIDA}") as traza:
        traza.start(delay=5)
        for _, c in consultas.iterrows():
            for modo in MODOS:
                for rep in range(1, REPETICIONES + 1):
                    m = marca(c["id"], rep, modo)
                    if modo == "frio":
                        labs.clear_cache(dataset=MODELO, workspace=WORKSPACE, verbose=False)
                    try:
                        res = fabric.evaluate_dax(MODELO, marcar(c["dax"], m), workspace=WORKSPACE)
                        ejecuciones.append((c["id"], modo, rep, m, len(res), huella_resultado(res), ""))
                    except Exception as e:
                        ejecuciones.append((c["id"], modo, rep, m, None, None, str(e)[:500]))
                    time.sleep(PAUSA_SEG)
            print("✓", c["id"], c["nombre"])
        time.sleep(SEG_ESPERA_TRAZA)
        logs = traza.stop(timeout=30)

ej = pd.DataFrame(ejecuciones, columns=["id", "modo", "rep", "marca", "filas", "huella", "error"])
print(f"{len(logs)} eventos de traza · {ej['error'].ne('').sum()} ejecuciones con error")

## 3. Calcular métricas y guardar

In [ ]:
met = metricas_de_traza(logs, ej["marca"].tolist())
mediciones = ej.merge(met, on="marca", how="left")
mediciones["encontrada"] = mediciones["encontrada"].fillna(False)
no_encontradas = mediciones[(mediciones["error"] == "") & (~mediciones["encontrada"])]
if len(no_encontradas):
    print(f"Atención: {len(no_encontradas)} ejecuciones sin eventos en la traza. "
          "Subí SEG_ESPERA_TRAZA o el delay de traza.start().")

resumen = resumir(mediciones)
mediciones.to_csv(f"{salida}/mediciones.csv", index=False, encoding="utf-8")
resumen.to_csv(f"{salida}/mediciones_resumen.csv", index=False, encoding="utf-8")
display(resumen.sort_values("total_ms_mediana", ascending=False))

inestables = resumen[~resumen["huella_estable"]]["id"].unique().tolist()
if inestables:
    print("Consultas con resultado distinto entre repeticiones (no sirven para comparar huella):", inestables)

## 4. Columnas que usan las consultas más lentas

`get_dax_query_dependencies` lista las columnas que toca cada consulta, con su tamaño en memoria. Le dice a Claude qué columnas conviene mirar primero.

In [ ]:
if DEPENDENCIAS_TOP:
    lentas = (resumen[resumen["modo"] == "frio"].sort_values("total_ms_mediana", ascending=False)
                     .head(DEPENDENCIAS_TOP)["id"].tolist())
    partes = []
    for i in lentas:
        dax = consultas.loc[consultas["id"] == i, "dax"].iloc[0]
        try:
            d = labs.get_dax_query_dependencies(dataset=MODELO, dax_string=dax, workspace=WORKSPACE)
            d.insert(0, "id", i)
            partes.append(d)
        except Exception as e:
            print(f"[{i}] sin dependencias:", e)
    if partes:
        pd.concat(partes).to_csv(f"{salida}/dependencias.csv", index=False, encoding="utf-8")

ahora = datetime.now(timezone.utc).isoformat(timespec="seconds")
ruta_corrida = f"{salida}/corrida.json"
corrida = json.load(open(ruta_corrida, encoding="utf-8")) if os.path.exists(ruta_corrida) else {}
corrida.update({"modelo": MODELO, "workspace": WORKSPACE, "corrida": CORRIDA, "commit": COMMIT or corrida.get("commit", ""),
                "mediciones_utc": ahora, "repeticiones": REPETICIONES, "modos": MODOS})
json.dump(corrida, open(ruta_corrida, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

if GUARDAR_DELTA:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ESQUEMA_DELTA}")
    d = mediciones.drop(columns=["marca"]).astype(str)
    d["modelo"], d["corrida"], d["commit"], d["fecha_utc"] = MODELO, CORRIDA, corrida["commit"], ahora
    spark.createDataFrame(d).write.mode("append").option("mergeSchema", "true").saveAsTable(f"{ESQUEMA_DELTA}.mediciones")

print(f"Listo en {salida}")